In [1]:
!pip install -q transformers datasets scikit-learn

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
import torch
import numpy as np

print("Done.")

Done.


In [2]:
# ── Cell 2: Data Loading + Label Coarsening ───────────────────────────────────

dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

# ── Politeness: 13 → 6 ───────────────────────────────────────────────────────
politeness_map = {
    "neutral":"neutral", "polite":"polite", "informal":"informal",
    "professional":"professional", "blunt":"blunt", "rude":"rude",
    "very_polite":"polite", "friendly":"informal", "religious":"polite",
    "impolite":"rude", "stern":"blunt", "sarcastic":"rude",
    "polite_but_firm":"polite"
}

# ── Tone: 131 → 5 (Appraisal Theory) ────────────────────────────────────────
positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal   = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal:    return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

# ── Power relation: 8 → 4 ────────────────────────────────────────────────────
power_map = {
    "equal":"equal",
    "inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior",
    "any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

# ── Apply all mappings ────────────────────────────────────────────────────────
def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

# ── Fit label encoders ────────────────────────────────────────────────────────
le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

print("Politeness classes:", list(le_pol.classes_))
print("Register classes:  ", list(le_reg.classes_))
print("Power classes:     ", list(le_pow.classes_))
print("Tone classes:      ", list(le_tone.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

burmese-conversational-pragmatics.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Politeness classes: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Register classes:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]
Power classes:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Tone classes:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]


In [3]:
# ── Cell 3: Encode Labels + Splits + Class Weights ────────────────────────────

def encode_all_labels(example):
    example["label_pol"]  = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_reg"]  = int(le_reg.transform([example["register_label"]])[0])
    example["label_pow"]  = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"] = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

# ── Same seed=42 split as all your previous notebooks ────────────────────────
split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

# ── Class weights ─────────────────────────────────────────────────────────────
def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

w_pol  = get_weights(train_data, "label_pol",  len(le_pol.classes_))
w_reg  = get_weights(train_data, "label_reg",  len(le_reg.classes_))
w_pow  = get_weights(train_data, "label_pow",  len(le_pow.classes_))
w_tone = get_weights(train_data, "label_tone", len(le_tone.classes_))

print("Weights pol: ", [round(x,3) for x in w_pol.tolist()])
print("Weights reg: ", [round(x,3) for x in w_reg.tolist()])
print("Weights pow: ", [round(x,3) for x in w_pow.tolist()])
print("Weights tone:", [round(x,3) for x in w_tone.tolist()])

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train: 1540  Val: 330  Test: 330
Weights pol:  [5.238, 0.976, 0.318, 0.817, 3.889, 6.111]
Weights reg:  [0.364, 4.01, 3.565, 1.39]
Weights pow:  [3.377, 0.33, 2.567, 3.5]
Weights tone: [1.149, 1.257, 4.738, 0.487, 0.936]


In [4]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device: cpu


In [5]:
# ── Load all models from HuggingFace ─────────────────────────────────────────
print("Loading Classifier A (register)...")
model_A = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-register").to(device).eval()

print("Loading Classifier B_v2 (power)...")
model_B = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-power-v2").to(device).eval()

print("Loading Classifier C_v2 (tone)...")
model_C = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-tone-v2").to(device).eval()

print("Loading Stage 3 politeness model...")
model_pol = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-pragmatics-stage3-v2").to(device).eval()

print("All models loaded.")

Loading Classifier A (register)...


config.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Classifier B_v2 (power)...


...

Loading Stage 3 politeness model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

All models loaded.


In [6]:
# ── Single-example inference helper ──────────────────────────────────────────

def predict_single(model, text, max_length=128):
    enc = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.argmax(logits, dim=1).item()

print("Helper ready.")

Helper ready.


In [7]:
# ── Stage 4 v2 — full chained pipeline ───────────────────────────────────────

reg_classes  = list(le_reg.classes_)   # A output
pow_classes  = list(le_pow.classes_)   # B output
tone_classes = list(le_tone.classes_)  # C output
pol_classes  = list(le_pol.classes_)   # final output

predictions_stage4v2 = []
true_labels          = []

print(f"Running chained pipeline on {len(test_data)} examples...")

for i, example in enumerate(test_data):
    utterance   = example["utterance"]
    context     = str(example["context"]).strip()
    instruction = str(example["instruction"]).strip()

    # ── Step 1: Classifier A → predicted register ─────────────────────────
    text_A   = utterance
    pred_reg = reg_classes[predict_single(model_A, text_A)]

    # ── Step 2: Classifier B_v2 → predicted power ─────────────────────────
    text_B   = f"[register: {pred_reg}] {utterance} </s> {context}"
    pred_pow = pow_classes[predict_single(model_B, text_B)]

    # ── Step 3: Classifier C_v2 → predicted tone ──────────────────────────
    text_C    = f"[register: {pred_reg}] [power: {pred_pow}] {utterance} </s> {context}"
    pred_tone = tone_classes[predict_single(model_C, text_C)]

    # ── Step 4: Stage 3 politeness model → final prediction ───────────────
    text_pol = (f"[register: {pred_reg}] [power: {pred_pow}] [tone: {pred_tone}] "
                f"{utterance} </s> {context} </s> {instruction}")
    pred_pol = pol_classes[predict_single(model_pol, text_pol)]

    predictions_stage4v2.append(pred_pol)
    true_labels.append(pol_classes[example["label_pol"]])

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/330 done...")

print("Done.")

Running chained pipeline on 330 examples...
  50/330 done...
  100/330 done...
  150/330 done...
  200/330 done...
  250/330 done...
  300/330 done...
Done.


In [8]:
# ── Evaluate Stage 4 v2 ───────────────────────────────────────────────────────

CLASSES = list(le_pol.classes_)

acc      = accuracy_score(true_labels, predictions_stage4v2)
macro_f1 = f1_score(true_labels, predictions_stage4v2, average="macro",
                    labels=CLASSES, zero_division=0)
w_f1     = f1_score(true_labels, predictions_stage4v2, average="weighted",
                    labels=CLASSES, zero_division=0)

print("── Stage 4 v2 (Codependent Chain) Results ───────────────────────")
print(f"Accuracy:    {acc:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {w_f1:.4f}")
print()
print(classification_report(true_labels, predictions_stage4v2,
                            labels=CLASSES, zero_division=0))

print("\n── Full Comparison ──────────────────────────────────────────────")
print(f"{'Model':<45} {'Macro F1':>10}")
print("-" * 57)
print(f"{'Stage 1 (utterance only)':<45} {'0.7014':>10}")
print(f"{'Stage 2 (+ context)':<45} {'0.7583':>10}")
print(f"{'Stage 3 (oracle metadata)':<45} {'0.7990':>10}")
print(f"{'Stage 4 original (predicted metadata)':<45} {'0.7020':>10}")
print(f"{'Stage 4 v2 (codependent chain)':<45} {macro_f1:>10.4f}")
print(f"{'ChatGPT o4-mini (zero-shot)':<45} {'0.6346':>10}")
print(f"{'Gemini 3 Fast (zero-shot)':<45} {'0.4929':>10}")

── Stage 4 v2 (Codependent Chain) Results ───────────────────────
Accuracy:    0.7242
Macro F1:    0.6957
Weighted F1: 0.7303

              precision    recall  f1-score   support

       blunt       1.00      0.57      0.73        14
    informal       0.66      0.66      0.66        59
     neutral       0.88      0.71      0.79       174
      polite       0.52      0.84      0.64        56
professional       0.71      1.00      0.83        15
        rude       0.55      0.50      0.52        12

    accuracy                           0.72       330
   macro avg       0.72      0.71      0.70       330
weighted avg       0.77      0.72      0.73       330


── Full Comparison ──────────────────────────────────────────────
Model                                           Macro F1
---------------------------------------------------------
Stage 1 (utterance only)                          0.7014
Stage 2 (+ context)                               0.7583
Stage 3 (oracle metadata)         